# NoiseBridge + mBERT — Kaggle Notebook

**Model:** NoiseBridge (PWNIC + GRL) with mBERT backbone  
**Runs:** clean, low, medium, high  

**Setup:**
1. Kaggle → Settings → Accelerator → **GPU T4 x1**
2. Add your `hindimix-data` dataset (Add Data → Your Datasets)
3. Run all cells

**Output:** `/kaggle/working/noisebridge_mbert_results.zip`

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.40.0", "jellyfish", "sentencepiece", "accelerate"], check=True)

import torch
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

import jellyfish
print('jellyfish: OK -', jellyfish.jaro_winkler_similarity('test', 'test'))

In [ ]:
import os

DATASET_NAME = 'hindimix-data'
DATA_DIR = f'/kaggle/input/{DATASET_NAME}'
if not os.path.exists(DATA_DIR):
    for d in os.listdir('/kaggle/input/'):
        if 'hindimix' in d.lower() or 'train' in d.lower():
            DATA_DIR = f'/kaggle/input/{d}'
            break

RESULTS_DIR = '/kaggle/working/results'
CKPT_DIR    = '/kaggle/working/checkpoints'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

print('Data dir:', DATA_DIR)
print('Files:')
for f in sorted(os.listdir(DATA_DIR)):
    print(f'  {f}')

## NoiseBridge Model (PWNIC + GRL)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function
from transformers import AutoModel
import jellyfish


def phonetic_similarity(text_a, text_b):
    words_a = str(text_a).lower().split()
    words_b = str(text_b).lower().split()
    if not words_a or not words_b:
        return 1.0
    n = max(len(words_a), len(words_b))
    scores = []
    for i in range(min(len(words_a), len(words_b))):
        try:
            scores.append(jellyfish.jaro_winkler_similarity(words_a[i], words_b[i]))
        except:
            scores.append(1.0)
    scores += [0.0] * (n - len(scores))
    return sum(scores) / n

def phonetic_weights(clean_texts, noisy_texts):
    weights = [1.0 - phonetic_similarity(c, n) for c, n in zip(clean_texts, noisy_texts)]
    return torch.tensor(weights, dtype=torch.float32)


class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.clone()
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None

class GradientReversalLayer(nn.Module):
    def __init__(self, lambda_max=0.5):
        super().__init__()
        self.lambda_max     = lambda_max
        self.current_lambda = 0.0
    def set_lambda(self, p):
        if p < 0.2:
            self.current_lambda = 0.0
        else:
            self.current_lambda = self.lambda_max * (p - 0.2) / 0.8
    def forward(self, x):
        return GradientReversalFunction.apply(x, self.current_lambda)


class PWNICLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature
    def forward(self, z_clean, z_noisy, phi):
        N      = z_clean.size(0)
        device = z_clean.device
        z_clean = F.normalize(z_clean.float(), dim=1)
        z_noisy = F.normalize(z_noisy.float(), dim=1)
        z_all   = torch.cat([z_clean, z_noisy], dim=0)
        sim = torch.mm(z_all, z_all.T) / self.temperature
        sim = sim.clamp(-15.0, 15.0)          # prevent fp16 exp overflow → inf → nan
        sim.masked_fill_(torch.eye(2*N, dtype=torch.bool, device=device), float('-inf'))
        pos_idx   = torch.cat([torch.arange(N, 2*N, device=device),
                                torch.arange(0, N,   device=device)])
        pos_sim   = sim[torch.arange(2*N, device=device), pos_idx]
        log_denom = torch.logsumexp(sim, dim=1)
        loss_each = (-(pos_sim - log_denom)).clamp(max=50.0)  # guard 0*inf=nan
        phi       = phi.float().to(device)
        phi_both  = torch.cat([phi, phi], dim=0)
        return (phi_both * loss_each).sum() / phi_both.sum().clamp(min=1e-8)


class NoiseAwareAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.gate = nn.Linear(hidden_size, 1)
    def forward(self, hidden_states):
        return hidden_states * torch.sigmoid(self.gate(hidden_states))


class NoiseBridge(nn.Module):
    def __init__(self, encoder_name, num_labels=2, num_noise_levels=4,
                 dropout=0.1, temperature=0.07,
                 alpha=0.5, beta=0.3, gamma=0.1, lambda_max=0.5,
                 class_weight=None):
        super().__init__()
        self.alpha = alpha
        self.beta  = beta
        self.gamma = gamma
        self.register_buffer('class_weight',
                             class_weight if class_weight is not None
                             else torch.ones(num_labels))
        self.encoder     = AutoModel.from_pretrained(encoder_name)
        self.hidden_size = self.encoder.config.hidden_size
        self.noise_attention     = NoiseAwareAttention(self.hidden_size)
        self.grl                 = GradientReversalLayer(lambda_max=lambda_max)
        self.dropout             = nn.Dropout(dropout)
        self.classifier          = nn.Sequential(
            nn.Linear(self.hidden_size, 256), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(256, num_labels)
        )
        self.adv_noise_predictor = nn.Sequential(
            nn.Linear(self.hidden_size, 128), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(128, num_noise_levels)
        )
        self.aux_noise_predictor = nn.Sequential(
            nn.Linear(self.hidden_size, 64), nn.ReLU(),
            nn.Linear(64, num_noise_levels)
        )
        self.pwnic = PWNICLoss(temperature=temperature)

    def set_grl_lambda(self, p):
        self.grl.set_lambda(p)

    def encode(self, input_ids, attention_mask):
        out    = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden = self.noise_attention(out.last_hidden_state)
        mask   = attention_mask.unsqueeze(-1).float()
        return (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

    def forward(self, input_ids, attention_mask, labels=None,
                noisy_input_ids=None, noisy_attention_mask=None,
                phi=None, noise_level_labels=None):
        z_clean = self.encode(input_ids, attention_mask)
        logits  = self.classifier(self.dropout(z_clean))
        loss_ce = F.cross_entropy(logits, labels, weight=self.class_weight) \
                  if labels is not None else None
        total_loss = loss_ce
        loss_pwnic = loss_adv = loss_aux = None
        if noisy_input_ids is not None:
            z_noisy = self.encode(noisy_input_ids, noisy_attention_mask)
            if phi is not None:
                loss_pwnic = self.pwnic(z_clean, z_noisy, phi)
                total_loss = total_loss + self.alpha * loss_pwnic
            if noise_level_labels is not None:
                unique_nl = noise_level_labels.unique()
                if len(unique_nl) > 1:
                    z_noisy_rev = self.grl(z_noisy)
                    adv_logits  = self.adv_noise_predictor(z_noisy_rev)
                    loss_adv    = F.cross_entropy(adv_logits, noise_level_labels)
                    total_loss  = total_loss + self.beta * loss_adv
                    aux_logits  = self.aux_noise_predictor(z_clean)
                    loss_aux    = F.cross_entropy(aux_logits, noise_level_labels)
                    total_loss  = total_loss + self.gamma * loss_aux
        return {'loss': total_loss, 'loss_ce': loss_ce,
                'loss_pwnic': loss_pwnic, 'loss_adv': loss_adv,
                'loss_aux': loss_aux, 'logits': logits}


print('NoiseBridge (mBERT) model defined.')

## Dataset & Training Utilities

In [ ]:
import json
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from sklearn.metrics import f1_score, classification_report
from tqdm.notebook import tqdm

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ENCODER    = "bert-base-multilingual-cased"
EPOCHS     = 5
BATCH_SIZE = 8
MAX_LEN    = 128
LR         = 3e-5
ALPHA      = 0.15
BETA       = 0.3
GAMMA      = 0.1
LAMBDA_MAX = 0.5

NOISE_LEVEL_MAP = {"clean": 0, "low": 1, "medium": 2, "high": 3}

print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


def compute_class_weights(labels):
    counts  = torch.bincount(torch.tensor(labels), minlength=2).float()
    weights = counts.sum() / (2.0 * counts.clamp(min=1))
    return weights


class NoiseBridgeDataset(Dataset):
    def __init__(self, clean_df, noisy_df, tokenizer):
        noisy_lookup = {}
        for _, row in noisy_df.iterrows():
            orig = str(row.get("text_original", ""))
            if orig and orig != "nan":
                noisy_lookup[orig] = (str(row["text"]),
                                      NOISE_LEVEL_MAP.get(str(row["noise_level"]).lower(), 1))

        self.clean_texts  = clean_df["text"].astype(str).tolist()
        self.labels       = clean_df["label"].astype(int).tolist()
        self.noisy_texts  = []
        self.noise_levels = []
        self.has_pair     = []

        for t in self.clean_texts:
            if t in noisy_lookup:
                nt, nl = noisy_lookup[t]
                self.noisy_texts.append(nt)
                self.noise_levels.append(nl)
                self.has_pair.append(True)
            else:
                self.noisy_texts.append(t)
                self.noise_levels.append(0)
                self.has_pair.append(False)

        self.tokenizer    = tokenizer
        self.class_weight = compute_class_weights(self.labels)
        n_pairs = sum(self.has_pair)
        n0 = self.labels.count(0); n1 = self.labels.count(1); total = len(self.labels)
        print(f"  CE samples: {total:,} | label 0: {n0:,} | label 1: {n1:,}")
        print(f"  Pairs: {n_pairs:,}/{total:,} | class_weight: {self.class_weight.tolist()}")
        print(f"  Computing phonetic weights...")
        self.phi = phonetic_weights(self.clean_texts, self.noisy_texts)
        for i, has in enumerate(self.has_pair):
            if not has:
                self.phi[i] = 0.0

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        c = self.tokenizer(self.clean_texts[idx], max_length=MAX_LEN,
                           padding="max_length", truncation=True, return_tensors="pt")
        n = self.tokenizer(self.noisy_texts[idx], max_length=MAX_LEN,
                           padding="max_length", truncation=True, return_tensors="pt")
        return {
            "input_ids":            c["input_ids"].squeeze(0),
            "attention_mask":       c["attention_mask"].squeeze(0),
            "noisy_input_ids":      n["input_ids"].squeeze(0),
            "noisy_attention_mask": n["attention_mask"].squeeze(0),
            "labels":               torch.tensor(self.labels[idx], dtype=torch.long),
            "noise_level_labels":   torch.tensor(self.noise_levels[idx], dtype=torch.long),
            "phi":                  self.phi[idx],
        }


class EvalDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=MAX_LEN,
                             padding="max_length", truncation=True, return_tensors="pt")
        return {"input_ids":      enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "labels":         torch.tensor(self.labels[idx], dtype=torch.long)}


def collate_fn(batch):
    return {k: torch.stack([b[k] for b in batch]) for k in batch[0]}


def load_data(noise_level):
    import gc
    train_df    = pd.read_csv(f"{DATA_DIR}/train.csv").dropna(subset=["text", "label"])
    val_df      = pd.read_csv(f"{DATA_DIR}/val.csv").dropna(subset=["text", "label"])
    clean_train = train_df[train_df["noise_level"] == "clean"].copy().drop_duplicates("text")
    pair_level  = "low" if noise_level == "clean" else noise_level
    noisy_train = train_df[train_df["noise_level"] == pair_level].copy()
    del train_df; gc.collect()
    test_path = f"{DATA_DIR}/test_clean.csv" if noise_level == "clean" \
                else f"{DATA_DIR}/test_noisy_{noise_level}.csv"
    test_df = pd.read_csv(test_path).dropna(subset=["text", "label"])
    print(f"  Clean: {len(clean_train):,} | Noisy ({pair_level}): {len(noisy_train):,} | Test: {len(test_df):,}")
    return clean_train, noisy_train, val_df, test_df


def train_epoch(model, loader, optimizer, scheduler, scaler, current_step, total_steps):
    model.train()
    total_loss = total_ce = total_pwnic = total_adv = total_aux = 0
    n_adv = 0
    for batch in tqdm(loader, desc="  train", leave=False):
        optimizer.zero_grad()
        model.set_grl_lambda(current_step / max(total_steps, 1))
        current_step += 1
        kwargs = {k: batch[k].to(DEVICE) for k in batch}
        with autocast("cuda"):
            out = model(**kwargs)
        scaler.scale(out["loss"]).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss  += out["loss"].item()
        total_ce    += out["loss_ce"].item()    if out["loss_ce"]    is not None else 0
        total_pwnic += out["loss_pwnic"].item() if out["loss_pwnic"] is not None else 0
        if out["loss_adv"] is not None:
            total_adv += out["loss_adv"].item(); n_adv += 1
        total_aux   += out["loss_aux"].item()   if out["loss_aux"]   is not None else 0
    nb = len(loader)
    return total_loss/nb, total_ce/nb, total_pwnic/nb, (total_adv/n_adv if n_adv else 0), total_aux/nb, current_step


def evaluate(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="  eval", leave=False):
            out = model(input_ids=batch["input_ids"].to(DEVICE),
                        attention_mask=batch["attention_mask"].to(DEVICE))
            preds.extend(out["logits"].argmax(dim=-1).cpu().tolist())
            targets.extend(batch["labels"].tolist())
    return f1_score(targets, preds, average="macro"), preds, targets


print(f"Utilities ready. Batch: {BATCH_SIZE} | fp16: True | ALPHA: {ALPHA}")

## Train NoiseBridge + mBERT — All 4 Noise Levels

In [ ]:
import gc
tokenizer   = AutoTokenizer.from_pretrained(ENCODER)
all_results = []

for noise in ['clean', 'low', 'medium', 'high']:
    print(f'\n{"="*65}')
    print(f'NoiseBridge+mBERT | noise={noise}')
    print(f'{"="*65}')

    clean_train, noisy_train, val_df, test_df = load_data(noise)

    train_ds = NoiseBridgeDataset(clean_train, noisy_train, tokenizer)
    val_ds   = EvalDataset(val_df['text'].tolist(), val_df['label'].tolist(), tokenizer)
    test_ds  = EvalDataset(test_df['text'].tolist(), test_df['label'].tolist(), tokenizer)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=collate_fn, num_workers=0)
    val_loader   = DataLoader(val_ds,  batch_size=BATCH_SIZE*2, collate_fn=collate_fn, num_workers=0)
    test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE*2, collate_fn=collate_fn, num_workers=0)

    del clean_train, noisy_train, val_df, test_df; gc.collect()

    model = NoiseBridge(
        encoder_name=ENCODER, alpha=ALPHA, beta=BETA,
        gamma=GAMMA, lambda_max=LAMBDA_MAX,
        class_weight=train_ds.class_weight,
    ).to(DEVICE)

    if hasattr(model.encoder, 'gradient_checkpointing_enable'):
        model.encoder.gradient_checkpointing_enable()
        print('  Gradient checkpointing: enabled')

    print(f'  Parameters: {sum(p.numel() for p in model.parameters()):,}')

    optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps  = len(train_loader) * EPOCHS
    scheduler    = get_linear_schedule_with_warmup(optimizer, total_steps//10, total_steps)
    scaler       = GradScaler('cuda')
    ckpt_path    = f'{CKPT_DIR}/noisebridge_mbert_{noise}.pt'
    best_val_f1  = 0.0
    current_step = 0

    for epoch in range(EPOCHS):
        loss, ce, pwnic, adv, aux, current_step = train_epoch(
            model, train_loader, optimizer, scheduler, scaler, current_step, total_steps
        )
        val_f1, _, _ = evaluate(model, val_loader)
        print(f'  Epoch {epoch+1}/{EPOCHS} | '
              f'loss:{loss:.4f} ce:{ce:.3f} pwnic:{pwnic:.3f} '
              f'adv:{adv:.3f} aux:{aux:.3f} | '
              f'val F1:{val_f1:.4f} | lambda:{model.grl.current_lambda:.3f}')
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), ckpt_path)
            print(f'    -> best saved')

    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    test_f1, test_preds, test_targets = evaluate(model, test_loader)
    print(f'\n  Test F1 (macro): {test_f1:.4f}')
    print(classification_report(test_targets, test_preds, target_names=['Non-hate', 'Hate']))

    result = {
        'model':         f'noisebridge_mbert_{noise}',
        'encoder':       ENCODER,
        'noise_level':   noise,
        'test_f1_macro': round(test_f1, 4),
        'best_val_f1':   round(best_val_f1, 4),
        'alpha': ALPHA, 'beta': BETA, 'gamma': GAMMA, 'lambda_max': LAMBDA_MAX,
        'epochs': EPOCHS, 'lr': LR, 'batch_size': BATCH_SIZE,
    }
    out = f'{RESULTS_DIR}/noisebridge_mbert_{noise}.json'
    with open(out, 'w') as f:
        json.dump(result, f, indent=2)
    print(f'  Saved -> {out}')
    all_results.append(result)

    del model; torch.cuda.empty_cache(); gc.collect()

print('\nAll done!')

## Summary + Download

In [ ]:
print('\n--- NoiseBridge + mBERT Results ---')
print(f'{"noise":<10} {"val F1":>8} {"test F1":>9}')
for r in all_results:
    print(f"  {r['noise_level']:<8} {r['best_val_f1']:>8.4f} {r['test_f1_macro']:>9.4f}")

import shutil
shutil.make_archive('/kaggle/working/noisebridge_mbert_results', 'zip', RESULTS_DIR)
print('\nZipped -> /kaggle/working/noisebridge_mbert_results.zip')
print('Download from Output tab ->')